# 05 Rain-lag extension (Step 8, pre-registered)

Does a more realistic rainfall effect raise the climate gain over B4s?

- **M1-R**: lags extended to 26 weeks; rain block mean split into wet = max(A, 0) and dry = min(A, 0) (24 features).
- **M1-RTH**: M1-R plus the same split for temperature and humidity (36 features).

The design was written in `results/rain_extension/PREREGISTRATION.md` before any new model was fitted. Results are precomputed; `RERUN = True` refits at 60 municipalities (about 6 min on 4 CPUs).

In [ ]:
# ---- Setup: works in Google Colab and locally (VS Code / Jupyter) ----
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aminah9945/dengue-climate-reconstruction"
BRANCH = "main"          # use "official-run" if the pull request is not merged yet
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/dengue-climate-reconstruction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all]"], check=True)
    # Optional: keep the downloaded data on Google Drive so it survives the session.
    USE_DRIVE = False
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ.setdefault("DATA_DIR", "/content/drive/MyDrive/dengue_recon_data")
else:
    # Local: this notebook lives in <repo>/notebooks/
    REPO = Path(os.environ.get("DENGUE_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()

os.environ.setdefault("DENGUE_ROOT", str(REPO))
os.environ.setdefault("DATA_DIR", str(REPO / "data"))
os.environ.setdefault("RESULTS_DIR", str(REPO / "results"))
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))      # makes the package importable without a kernel restart

from dengue_recon import config as C
C.ensure_dirs()
print("repo       :", C.ROOT)
print("DATA_DIR   :", C.DATA_DIR, "| processed data present:", (C.DATA_DIR / "dengue_weekly.parquet").exists())
print("RESULTS_DIR:", C.RESULTS_DIR)

In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200, "display.max_columns", 30)


def show(path, width=900):
    p = Path(path)
    p = p if p.is_absolute() else C.FIG_DIR / p
    display(Image(str(p), width=width)) if p.exists() else print("missing figure:", p)


def have_data():
    ok = (C.DATA_DIR / "dengue_weekly.parquet").exists()
    if not ok:
        print("Processed data not found in DATA_DIR - run notebook 00 first (or set DATA_DIR).")
    return ok

In [ ]:
RERUN = False
if RERUN and have_data():
    from dengue_recon.extensions import rain_lags
    rain_lags.main(60, ("M1", "M1-R", "M1-RTH"), "weekly_60")
    rain_lags.main(60, ("M1-R-shufTT-s0",), "weekly_60_shuf")
display(Markdown((C.RAIN_DIR / "PREREGISTRATION.md").read_text()[:1500] + " ..."))

In [ ]:
for s in (60, 300):
    t = pd.read_csv(C.RAIN_DIR / f"table_paired_weekly_{s}.csv")
    display(Markdown(f"**{s} municipalities**"))
    display(t[(t.metric == "jsd") & (t.subset == "all")][["model_a", "model_b", "n", "mean_diff", "rel_change", "ci_muni_lo", "ci_muni_hi", "ci_year_lo", "ci_year_hi", "n_years_a_better"]].round(5))

In [ ]:
show(C.RAIN_DIR / "fig_rain_gain_vs_M1_B4s.png"); show(C.RAIN_DIR / "fig_rain_coefficients_300.png", 1100)